# NB06 — Editor UX Latency

Uses **Playwright** to automate the QCanvas IDE and measure the UX responsiveness of the code editor:

| Metric | What it measures |
|--------|------------------|
| **Code → Compile turnaround** | Click Compile → result appears (ms) |
| **Monaco keystroke latency** | Time per keypress in the editor (ms) |
| **Result pane render time** | From API response to DOM update visible |
| **Page interaction readiness** | Time until Monaco editor accepts input |

**Requires:** Frontend running at `http://localhost:3000` AND `playwright install chromium`  
**Output:** `../results/frontend/editor_timing.csv`

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'scripts'))

try:
    from playwright.sync_api import sync_playwright
    PLAYWRIGHT_OK = True
    print('✓ Playwright available')
except ImportError:
    PLAYWRIGHT_OK = False
    print('✗ Playwright not installed. Run: pip install playwright && playwright install chromium')

import time
import statistics
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from playwright_helpers import (
    measure_editor_compile_turnaround,
    measure_editor_keystroke_latency,
    FRONTEND_URL,
) if PLAYWRIGHT_OK else (None, None, None)

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams.update({'figure.dpi': 130})

RESULTS_DIR = Path('../results/frontend')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

IDE_URL = f'{FRONTEND_URL}/app'
N_RUNS = 5  # repeat measurements for each test
print(f'IDE URL: {IDE_URL}')

## 1 — Page Interaction Readiness

How long after navigation does the IDE become usable (Monaco editor accepting input)?

In [ ]:
if not PLAYWRIGHT_OK:
    raise RuntimeError('Playwright required')

readiness_times = []

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    for run in range(N_RUNS):
        ctx = browser.new_context(viewport={'width': 1440, 'height': 900})
        page = ctx.new_page()
        t0 = time.perf_counter()
        try:
            page.goto(IDE_URL, timeout=30000, wait_until='networkidle')
            # Wait for Monaco editor to appear
            page.wait_for_selector('.monaco-editor', timeout=15000)
            ttir_ms = (time.perf_counter() - t0) * 1000
            readiness_times.append({'run': run+1, 'ttir_ms': ttir_ms, 'error': None})
            print(f'  Run {run+1}: TTIR = {ttir_ms:.0f}ms')
        except Exception as exc:
            ttir_ms = (time.perf_counter() - t0) * 1000
            readiness_times.append({'run': run+1, 'ttir_ms': None, 'error': str(exc)[:80]})
            print(f'  Run {run+1}: ERROR - {exc}')
        finally:
            ctx.close()
    browser.close()

df_readiness = pd.DataFrame(readiness_times)
valid_times = [r['ttir_ms'] for r in readiness_times if r['ttir_ms'] is not None]
if valid_times:
    print(f'\nTTIR: mean={statistics.mean(valid_times):.0f}ms  p95={sorted(valid_times)[int(0.95*len(valid_times))]:.0f}ms')

## 2 — Code → Compile Turnaround

Type code into Monaco, click Convert/Compile, measure time until results appear.

In [ ]:
TEST_CIRCUITS = [
    {
        'label': 'Cirq Bell (small)',
        'framework': 'cirq',
        'code': (
            'import cirq\n'
            'q0, q1 = cirq.LineQubit.range(2)\n'
            'circuit = cirq.Circuit([\n'
            '    cirq.H(q0),\n'
            '    cirq.CNOT(q0, q1),\n'
            '    cirq.measure(q0, q1, key="r")\n'
            '])'
        ),
    },
    {
        'label': 'Qiskit Grover (medium)',
        'framework': 'qiskit',
        'code': (
            'from qiskit import QuantumCircuit\n'
            'qc = QuantumCircuit(2, 2)\n'
            'qc.h([0, 1])\n'
            'qc.cz(0, 1)\n'
            'qc.h([0, 1])\n'
            'qc.x([0, 1])\n'
            'qc.cz(0, 1)\n'
            'qc.x([0, 1])\n'
            'qc.h([0, 1])\n'
            'qc.measure([0, 1], [0, 1])'
        ),
    },
]

turnaround_rows = []

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    for circuit in TEST_CIRCUITS:
        print(f'\nMeasuring: {circuit["label"]} ({N_RUNS} runs)...')
        for run in range(N_RUNS):
            ctx = browser.new_context(viewport={'width': 1440, 'height': 900})
            page = ctx.new_page()
            try:
                page.goto(IDE_URL, timeout=30000, wait_until='networkidle')
                page.wait_for_selector('.monaco-editor', timeout=15000)
                page.wait_for_timeout(1000)  # let Monaco fully initialise

                result = measure_editor_compile_turnaround(
                    page, circuit['code'], framework=circuit['framework']
                )
                result['label'] = circuit['label']
                result['run'] = run + 1
                turnaround_rows.append(result)
                ms_str = f"{result['turnaround_ms']:.0f}ms" if result['turnaround_ms'] else 'ERR'
                print(f'  Run {run+1}: {ms_str}')
            except Exception as exc:
                turnaround_rows.append({'label': circuit['label'], 'run': run+1, 'turnaround_ms': None, 'error': str(exc)[:80]})
                print(f'  Run {run+1}: ERROR - {exc}')
            finally:
                ctx.close()
    browser.close()

df_turnaround = pd.DataFrame(turnaround_rows)
display(df_turnaround)

## 3 — Monaco Keystroke Latency

In [ ]:
keystroke_rows = []

with sync_playwright() as pw:
    browser = pw.chromium.launch(headless=True)
    for run in range(N_RUNS):
        ctx = browser.new_context(viewport={'width': 1440, 'height': 900})
        page = ctx.new_page()
        try:
            page.goto(IDE_URL, timeout=30000, wait_until='networkidle')
            page.wait_for_selector('.monaco-editor', timeout=15000)
            page.wait_for_timeout(1000)

            result = measure_editor_keystroke_latency(page, n_chars=30)
            result['run'] = run + 1
            keystroke_rows.append(result)
            ms_str = f"{result['mean_keystroke_ms']:.1f}ms" if result['mean_keystroke_ms'] else 'ERR'
            print(f'Keystroke run {run+1}: mean={ms_str}')
        except Exception as exc:
            keystroke_rows.append({'run': run+1, 'mean_keystroke_ms': None, 'error': str(exc)[:80]})
            print(f'Keystroke run {run+1}: ERROR - {exc}')
        finally:
            ctx.close()
    browser.close()

df_keystroke = pd.DataFrame(keystroke_rows)
display(df_keystroke)

## 4 — Save and Plot All Editor Metrics

In [ ]:
# Combine and save
df_readiness['metric'] = 'TTIR (Time to IDE Ready)'
df_readiness['latency_ms'] = df_readiness['ttir_ms']

df_ta = df_turnaround.copy()
df_ta['metric'] = df_ta['label'].apply(lambda x: f'Compile Turnaround: {x}')
df_ta['latency_ms'] = df_ta['turnaround_ms']

df_ks = df_keystroke.copy()
df_ks['metric'] = 'Keystroke Latency (30 chars)'
df_ks['latency_ms'] = df_ks['mean_keystroke_ms']

combined = [
    ('TTIR (ms)', [r['ttir_ms'] for r in readiness_times if r['ttir_ms']]),
    *[
        (f'Compile: {c["label"]}',
         [r['turnaround_ms'] for r in turnaround_rows if r.get('label') == c['label'] and r.get('turnaround_ms')])
        for c in TEST_CIRCUITS
    ],
    ('Keystroke (ms)', [r['mean_keystroke_ms'] for r in keystroke_rows if r.get('mean_keystroke_ms')]),
]

summary_rows = []
for label, vals in combined:
    if vals:
        summary_rows.append({
            'metric': label,
            'mean_ms': statistics.mean(vals),
            'p95_ms': sorted(vals)[int(0.95 * len(vals))] if len(vals) > 1 else vals[0],
            'min_ms': min(vals),
            'max_ms': max(vals),
            'n': len(vals),
        })

df_editor = pd.DataFrame(summary_rows)
df_editor.to_csv(RESULTS_DIR / 'editor_timing.csv', index=False)
print('✓ Saved editor_timing.csv')
display(df_editor.round(1))

In [ ]:
if not df_editor.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    colors = ['steelblue' if 'TTIR' in m else ('orchid' if 'Keystroke' in m else 'tomato')
              for m in df_editor['metric']]
    ax.barh(df_editor['metric'], df_editor['mean_ms'], xerr=df_editor['p95_ms'] - df_editor['mean_ms'],
            color=colors, alpha=0.85, capsize=5)
    ax.axvline(3000, color='red', linestyle='--', linewidth=1, label='3s threshold')
    ax.axvline(1000, color='green', linestyle='--', linewidth=1, label='1s threshold')
    ax.set_xlabel('Time (ms)')
    ax.set_title('QCanvas IDE — Editor UX Timing (mean ± p95)')
    ax.legend(fontsize=9)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'editor_timing.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('✓ Saved editor_timing.png')

## 5 — Summary

In [ ]:
print('=' * 55)
print('EDITOR UX LATENCY SUMMARY')
print('=' * 55)
for _, row in df_editor.iterrows():
    rating = '✓ Good' if row['mean_ms'] < 1000 else ('~ OK' if row['mean_ms'] < 3000 else '✗ Slow')
    print(f"  {rating:8s}  {row['metric']}: {row['mean_ms']:.0f}ms (p95={row['p95_ms']:.0f}ms)")
print('=' * 55)